# TerraMind: Latent Response to the Degradation Ladder

Companion to `domain_gap_terramind.ipynb`. That notebook asks whether the gap between **real** $\Phi$-sat-2 and Sentinel-2B is visible in TerraMind's representation; this one asks **which simulated degradation factor produces it**, using the same measures on the graded ladder of Section~`sec:sen1floods-sim`.

The comparison is no longer across sensors but across rungs of the ladder: the undegraded (`clean`) variant is the reference, and every degraded variant is a paired view of the *same scene content*, differing only in the one factor being varied. That is the property the ladder was built for -- with real sensor pairs, several physical differences move at once and none can be attributed.

Scope: **frozen, pretrained `terramind_v1_base` only**. No fine-tuning, no student, no downstream head. The downstream counterpart of this analysis (flood IoU across the same ladder) is reported separately in the Results chapter; the point of doing both is to check whether the representation-level ordering of the factors matches the task-level one.

Three factors, four severities each, on a common axis where level 4 is the full simulator configuration and each step down halves the perturbation:

| Level | Severity | SNR range | PSF $\sigma$ (px) | Misalignment $\sigma$ (px) |
|---|---|---|---|---|
| L1 | 0.125 | [40, 80] | 0.5 | 1.25 |
| L2 | 0.25 | [20, 40] | 1.0 | 2.50 |
| L3 | 0.5 | [10, 20] | 2.0 | 5.00 |
| L4 | 1.0 | [5, 10] | 4.0 | 10.00 |

Each factor is analysed in its own cell, then compared across factors at the end.

## 1. Setup

Paths, the ladder definition, and the frozen encoder. Normalization follows the **flood pipeline** (plain per-band z-score on scaled reflectance), not the sqrt/clip convention of the triplet notebook, so these embeddings correspond to the inputs the flood experiments actually used.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
import torch.nn.functional as F
from rasterio.windows import Window
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC
from terratorch import BACKBONE_REGISTRY

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
N_SAMPLES = 300          # scenes drawn from the union of the three splits
CROP = 256               # centre crop in 4.75 m pixels; keeps 13 x N forwards tractable
BATCH = 8
SCENE_SIZE = 1077

LADDER_ROOT = Path("/shared/home/elucas/datasets/sen1floods11_ladder")
LABEL_ROOT = Path("/shared/home/elucas/datasets/sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand")
SPLIT_DIR = LADDER_ROOT / "clean/v1.1/splits/flood_handlabeled"
S2_SUBDIR = "v1.1/data/flood_events/HandLabeled/S2Hand"
FIG_DIR = "/shared/home/elucas/scratch/masters-thesis/figures"

FACTORS = {"snr": "Sensor noise (SNR)", "psf": "Optical blur (PSF)", "misalign": "Band misalignment"}
LEVELS = [1, 2, 3, 4]
SEVERITY = {0: 0.0, 1: 0.125, 2: 0.25, 3: 0.5, 4: 1.0}
VARIANTS = ["clean"] + [f"{f}_l{l}" for f in FACTORS for l in LEVELS]
FACTOR_COLOR = {"snr": "#d62728", "psf": "#2ca02c", "misalign": "#1f77b4"}
LEVEL_ALPHA = {1: 0.30, 2: 0.50, 3: 0.75, 4: 1.0}

print(f"{len(VARIANTS)} variants:", ", ".join(VARIANTS))

In [ ]:
# Band 3 of the simulated product is panchromatic and is dropped, leaving
# Blue, Green, Red, RE1, RE2, RE3, NIR -- the order TerraMind is given below.
BANDS = [0, 1, 2, 4, 5, 6, 7]
MEANS = np.array([2137.385, 2018.788, 2082.986, 2295.651, 2854.537, 3122.849, 3040.560], dtype=np.float32)
STDS = np.array([1675.806, 1557.708, 1833.702, 1823.738, 1733.977, 1732.131, 1679.732], dtype=np.float32)

_mean_t = torch.tensor(MEANS, device=device).view(1, -1, 1, 1)
_std_t = torch.tensor(STDS, device=device).view(1, -1, 1, 1)

OFF = (SCENE_SIZE - CROP) // 2
WINDOW = Window(OFF, OFF, CROP, CROP)


def normalize(img):
    """(B,7,H,W) scaled reflectance -> z-scored, matching the flood pipeline."""
    return (img - _mean_t) / _std_t


def read_crop(variant, scene_id):
    """Centre crop of one scene from one ladder variant, PAN dropped."""
    path = LADDER_ROOT / variant / S2_SUBDIR / f"simulated_L1C_{scene_id}_S2Hand.tif"
    with rasterio.open(path) as src:
        arr = src.read(window=WINDOW).astype(np.float32)
    return arr[BANDS]


def water_fraction(scene_id):
    """Water share of the labelled pixels under the same centre crop.

    Labels are 512 px over the ground that the imagery covers in 1077 px, so the
    crop is scaled rather than reused directly.
    """
    with rasterio.open(LABEL_ROOT / f"{scene_id}_LabelHand.tif") as src:
        lab = src.read(1)
    size = int(round(CROP * lab.shape[0] / SCENE_SIZE))
    off = (lab.shape[0] - size) // 2
    patch = lab[off:off + size, off:off + size]
    valid = patch >= 0
    return float((patch[valid] == 1).mean()) if valid.any() else np.nan


scene_ids = []
for split in ("train", "valid", "test"):
    scene_ids += [ln.strip() for ln in (SPLIT_DIR / f"flood_{split}_data.txt").read_text().splitlines() if ln.strip()]
scene_ids = sorted(set(scene_ids))

rng = np.random.default_rng(SEED)
if len(scene_ids) > N_SAMPLES:
    scene_ids = [scene_ids[i] for i in sorted(rng.choice(len(scene_ids), N_SAMPLES, replace=False))]
N = len(scene_ids)
print(f"{N} scenes; crop {CROP}x{CROP} at offset {OFF}")

In [ ]:
COMPARED_BANDS = {"S2L1C": {"B02": 0, "B03": 1, "B04": 2, "B05": 3, "B06": 4, "B07": 5, "B08": 6}}

encoder = (
    BACKBONE_REGISTRY.build("terramind_v1_base", pretrained=True, modalities=["S2L1C"], bands=COMPARED_BANDS)
    .to(device)
    .eval()
)
for p in encoder.parameters():
    p.requires_grad_(False)
print("TerraMind Base loaded (pretrained, frozen -- no fine-tuning applied).")

## 2. Extract per-layer embeddings for every rung

One forward pass per (variant, scene), mean-pooling tokens at each of the 12 blocks. Because every variant holds the same scenes in the same order, row $i$ of any two variants is a **paired** view of identical ground, which is what makes the cosine and probe measures below meaningful.

In [ ]:
CACHE = Path(f"/shared/home/elucas/scratch/terra-sat-drift/outputs/ladder_embeddings_n{N_SAMPLES}_c{CROP}_s{SEED}.npz")

t0 = time.time()
layer_embs = {}

if CACHE.exists():
    with np.load(CACHE) as z:
        layer_embs = {k: z[k] for k in z.files}
    print(f"loaded cached embeddings from {CACHE.name}")

for variant in [v for v in VARIANTS if v not in layer_embs]:
    chunks = []
    for start in range(0, N, BATCH):
        batch_ids = scene_ids[start:start + BATCH]
        arr = np.stack([read_crop(variant, sid) for sid in batch_ids])
        x = normalize(torch.from_numpy(arr).to(device))
        with torch.no_grad():
            layers = [layer.mean(dim=1).float().cpu().numpy() for layer in encoder(x)]
        chunks.append(np.stack(layers, axis=0))          # (n_layers, batch, dim)
    layer_embs[variant] = np.concatenate(chunks, axis=1)  # (n_layers, N, dim)
    print(f"  {variant:<14} {layer_embs[variant].shape}  ({time.time() - t0:.0f}s)")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(CACHE, **layer_embs)
    print(f"cached to {CACHE}")

n_layers = layer_embs["clean"].shape[0]
final = n_layers - 1
print(f"\n{n_layers} layers, dim {layer_embs['clean'].shape[-1]}, {time.time() - t0:.0f}s total")

In [ ]:
# Patch-level target for the probe-transfer test. A dominant-class label is unusable
# here because water is a small minority of pixels in most scenes, so the split is
# taken at the median water fraction, which balances the two classes by construction.
frac = np.array([water_fraction(sid) for sid in scene_ids])
ok = ~np.isnan(frac)
y_water = (frac > np.nanmedian(frac)).astype(int)
print(f"water fraction: median {np.nanmedian(frac):.3f}, "
      f"class balance {y_water[ok].mean():.2f}, {ok.sum()}/{N} scenes labelled")

## 3. Measures

The same four instruments as the sensor-gap notebook, each rewritten to take an arbitrary pair of variants:

- **Paired cosine similarity** between the clean and degraded views of the same scene.
- **Per-layer cosine**, to locate where in the encoder a factor first bites.
- **Proxy $\mathcal{A}$-distance**, the separability of clean from degraded under a linear probe: PAD $\rightarrow 2$ means the degradation is trivially detectable in the representation, PAD $\rightarrow 0$ means it leaves no linear trace.
- **Probe transfer**, fitting a linear classifier on clean embeddings and applying it to degraded embeddings of the same scenes. This is the functional question, and the one that should track the downstream IoU result.

In [ ]:
def paired_cosine(a, b):
    return F.cosine_similarity(torch.from_numpy(a), torch.from_numpy(b), dim=1).numpy()


def pad_score(a, b, cv=5, permute=False):
    """Proxy A-distance between two sets of embeddings.

    With ``permute`` the domain labels are shuffled first, which gives the null
    level of the measure. That control matters here: a linear SVM in 768 dimensions
    separates a few hundred points almost regardless of content, so PAD saturates
    toward 2 when the sample is small relative to the embedding dimension. Read the
    reported PAD against its own null, not against the theoretical 0.
    """
    X = np.concatenate([a, b], axis=0)
    y = np.concatenate([np.zeros(len(a)), np.ones(len(b))])
    if permute:
        y = np.random.default_rng(SEED).permutation(y)
    err = 1.0 - cross_val_score(LinearSVC(max_iter=5000, dual="auto"), X, y, cv=cv).mean()
    return 2 * (1 - 2 * err)


def probe_transfer(variant, train_idx, test_idx):
    """Fit on clean, score on clean and on the degraded variant (same held-out scenes)."""
    Xc = layer_embs["clean"][final][ok]
    Xd = layer_embs[variant][final][ok]
    y = y_water[ok]
    probe = LogisticRegression(max_iter=2000).fit(Xc[train_idx], y[train_idx])
    return probe.score(Xc[test_idx], y[test_idx]), probe.score(Xd[test_idx], y[test_idx])


_perm = rng.permutation(int(ok.sum()))
_split = int(0.7 * len(_perm))
TRAIN_IDX, TEST_IDX = _perm[:_split], _perm[_split:]


def analyse_factor(factor, save=True):
    """Run all four measures over the four rungs of one factor and plot them."""
    name = FACTORS[factor]
    sev = [SEVERITY[l] for l in LEVELS]
    rows, per_layer = [], {}

    for lvl in LEVELS:
        v = f"{factor}_l{lvl}"
        cos_f = paired_cosine(layer_embs["clean"][final], layer_embs[v][final])
        per_layer[lvl] = [paired_cosine(layer_embs["clean"][k], layer_embs[v][k]).mean() for k in range(n_layers)]
        pad = pad_score(layer_embs["clean"][final], layer_embs[v][final])
        pad_null = pad_score(layer_embs["clean"][final], layer_embs[v][final], permute=True)
        acc_c, acc_d = probe_transfer(v, TRAIN_IDX, TEST_IDX)
        rows.append(dict(level=lvl, severity=SEVERITY[lvl], cos_mean=cos_f.mean(), cos_std=cos_f.std(),
                         pad=pad, pad_null=pad_null, probe_clean=acc_c, probe_deg=acc_d,
                         retention=acc_d / acc_c))

    print(f"\n{name}")
    print(f"{'level':<7}{'severity':>9}{'cosine':>18}{'PAD':>8}{'(null)':>8}"
          f"{'probe clean':>13}{'probe deg':>11}{'retention':>11}")
    for r in rows:
        print(f"L{r['level']:<6}{r['severity']:>9.3f}{r['cos_mean']:>11.4f}+-{r['cos_std']:<5.3f}"
              f"{r['pad']:>8.3f}{r['pad_null']:>8.3f}"
              f"{r['probe_clean']:>13.3f}{r['probe_deg']:>11.3f}{r['retention']:>10.1%}")

    c = FACTOR_COLOR[factor]
    fig, ax = plt.subplots(2, 2, figsize=(12, 8.5))
    fig.suptitle(f"TerraMind Base latent response: {name}", fontsize=13)

    m = np.array([r["cos_mean"] for r in rows]); s = np.array([r["cos_std"] for r in rows])
    ax[0, 0].plot(sev, m, marker="o", color=c)
    ax[0, 0].fill_between(sev, m - s, m + s, alpha=0.15, color=c)
    ax[0, 0].set_xlabel("Severity"); ax[0, 0].set_ylabel("Paired cosine vs clean (final layer)")
    ax[0, 0].set_title("Displacement of the paired view"); ax[0, 0].set_ylim(0, 1.02)

    for lvl in LEVELS:
        ax[0, 1].plot(range(1, n_layers + 1), per_layer[lvl], marker="o", ms=3.5,
                      color=c, alpha=LEVEL_ALPHA[lvl], label=f"L{lvl} (sev {SEVERITY[lvl]})")
    ax[0, 1].set_xlabel("Encoder block"); ax[0, 1].set_ylabel("Mean paired cosine")
    ax[0, 1].set_title("Where in the encoder the factor bites"); ax[0, 1].legend(fontsize=8)

    ax[1, 0].plot(sev, [r["pad"] for r in rows], marker="s", color=c)
    ax[1, 0].plot(sev, [r["pad_null"] for r in rows], marker="x", ls="--", lw=1,
                  color="0.6", label="permuted-label null")
    ax[1, 0].axhline(2.0, color="gray", ls=":", lw=1, label="max separability")
    ax[1, 0].set_xlabel("Severity"); ax[1, 0].set_ylabel("Proxy $\\mathcal{A}$-distance")
    ax[1, 0].set_title("Linear detectability of the degradation"); ax[1, 0].set_ylim(0, 2.05)
    ax[1, 0].legend(fontsize=8)

    ax[1, 1].plot(sev, [r["retention"] for r in rows], marker="^", color=c)
    ax[1, 1].axhline(1.0, color="gray", ls=":", lw=1)
    ax[1, 1].set_xlabel("Severity"); ax[1, 1].set_ylabel("Probe accuracy retained")
    ax[1, 1].set_title("Functional transfer clean $\\rightarrow$ degraded"); ax[1, 1].set_ylim(0, 1.1)

    for a in ax.ravel():
        a.grid(True, ls="--", alpha=0.4)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if save:
        fig.savefig(f"{FIG_DIR}/fig_ladder_latent_{factor}.png", dpi=150)
    plt.show()
    return rows

## 4. Sensor noise (SNR)

The factor that dominates the downstream result. Base SNR is drawn per scene from the stated range and noise amplitude scales as its reciprocal, so the four rungs correspond to relative noise of $1/8$, $1/4$, $1/2$, $1$.

In [ ]:
res_snr = analyse_factor("snr")

## 5. Optical blur (PSF)

A Gaussian point-spread function of increasing width, applied per band. This is the factor that models a resolution difference, and the one the downstream evaluation found nearly free.

In [ ]:
res_psf = analyse_factor("psf")

## 6. Band misalignment

Random per-band sub-pixel shifts, the factor whose downstream cost proved most reproducible across models. It is the only one of the three that breaks the *cross-band* relationships rather than acting within each band independently.

In [ ]:
res_mis = analyse_factor("misalign")

## 7. Factors side by side

All three on one severity axis. The question this answers is whether the representation ranks the factors the same way the downstream flood IoU does; if it does, the latent measures are a cheap proxy for an expensive fine-tuning sweep.

**Expect a disagreement here, and read it as a property of the instrument rather than of the factor.** These embeddings are token means, so a perturbation that moves pixels around without changing their distribution -- band misalignment above all, optical blur to a lesser extent -- is substantially averaged out by the pooling, while sensor noise shifts the per-band statistics that survive it. The downstream evaluation found band misalignment to be the most reproducible source of IoU loss; if the pooled latent measures rank it far below noise, that is evidence that mean-pooled patch embeddings are insensitive to geometric degradation, not evidence that misalignment is harmless. A token-level or spatially resolved measure would be the right follow-up for the geometric factors.

In [ ]:
results = {"snr": res_snr, "psf": res_psf, "misalign": res_mis}
sev = [SEVERITY[l] for l in LEVELS]

fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
for key, title, ylab, lim in [("cos_mean", "Paired cosine vs clean", "Cosine (final layer)", (0, 1.02)),
                              ("pad", "Proxy $\\mathcal{A}$-distance", "PAD", (0, 2.05)),
                              ("retention", "Probe transfer retention", "Accuracy retained", (0, 1.1))]:
    i = ["cos_mean", "pad", "retention"].index(key)
    for f, rows in results.items():
        ax[i].plot(sev, [r[key] for r in rows], marker="o", color=FACTOR_COLOR[f], label=FACTORS[f])
    ax[i].set_title(title); ax[i].set_xlabel("Severity"); ax[i].set_ylabel(ylab)
    ax[i].set_ylim(*lim); ax[i].grid(True, ls="--", alpha=0.4); ax[i].legend(fontsize=8)
fig.suptitle("TerraMind Base: latent response to each degradation factor", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(f"{FIG_DIR}/fig_ladder_latent_all_factors.png", dpi=150)
plt.show()

print(f"\nAt full severity (L4):\n{'factor':<20}{'cosine':>9}{'PAD':>8}{'retention':>11}")
for f, rows in results.items():
    r = rows[-1]
    print(f"{FACTORS[f]:<20}{r['cos_mean']:>9.4f}{r['pad']:>8.3f}{r['retention']:>10.1%}")

### Per-layer separability at full severity

Where each factor becomes linearly detectable. A factor visible from the first block is low-level; one that only emerges deeper has been introduced by the encoder's own processing.

In [ ]:
fig, axp = plt.subplots(figsize=(9.5, 5))
for f in FACTORS:
    v = f"{f}_l4"
    pads = [pad_score(layer_embs["clean"][k], layer_embs[v][k]) for k in range(n_layers)]
    axp.plot(range(1, n_layers + 1), pads, marker="o", color=FACTOR_COLOR[f], label=FACTORS[f])
    print(f"{FACTORS[f]:<20}" + " ".join(f"{p:.2f}" for p in pads))
axp.axhline(2.0, color="gray", ls=":", lw=1, label="max separability")
axp.set_xlabel("Encoder block"); axp.set_ylabel("Proxy $\\mathcal{A}$-distance (clean vs L4)")
axp.set_title("Per-layer detectability of each factor at full severity")
axp.set_ylim(0, 2.05); axp.grid(True, ls="--", alpha=0.4); axp.legend(fontsize=9)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_ladder_latent_pad_per_layer.png", dpi=150)
plt.show()

## 8. Projections

Final-layer embeddings for one factor at a time, clean plus its four rungs. A factor that merely translates the cloud rigidly is a nuisance shift a normalization could absorb; one that scatters or folds it has changed the representation's structure.

In [ ]:
def project(reducer_name="umap"):
    fig, axs = plt.subplots(1, 3, figsize=(16, 5.2))
    for i, f in enumerate(FACTORS):
        variants = ["clean"] + [f"{f}_l{l}" for l in LEVELS]
        X = np.concatenate([layer_embs[v][final] for v in variants], axis=0)
        if reducer_name == "umap":
            import umap
            Z = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(X)
        else:
            Z = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca",
                     learning_rate="auto").fit_transform(X)
        for j, v in enumerate(variants):
            sl = slice(j * N, (j + 1) * N)
            lvl = 0 if v == "clean" else int(v[-1])
            axs[i].scatter(Z[sl, 0], Z[sl, 1], s=9, alpha=0.55,
                           color="0.55" if lvl == 0 else FACTOR_COLOR[f],
                           edgecolors="none",
                           label="clean" if lvl == 0 else f"L{lvl}")
            if lvl:
                axs[i].collections[-1].set_alpha(0.25 + 0.18 * lvl)
        axs[i].set_title(FACTORS[f]); axs[i].legend(fontsize=7, markerscale=1.6)
        axs[i].grid(True, ls="--", alpha=0.3)
    fig.suptitle(f"{reducer_name.upper()} of final-layer embeddings, clean vs each rung ({N} scenes)", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(f"{FIG_DIR}/fig_ladder_latent_{reducer_name}.png", dpi=150)
    plt.show()


project("umap")

In [ ]:
project("tsne")